# Getting started

A five-minute tour of your appliance workspace: the local warehouse, the shared lake, BigQuery and the other services you may have running.

Run the cells top to bottom (Shift+Enter). Nothing here changes data.

## 1. The local warehouse (`bsp_edw`)

The synthetic ISP dataset lives in the `raw` schema; dbt builds `staging → intermediate → marts`.

In [ ]:
from bsp_common import db

db.query_df("""
    select table_schema, count(*) as tables
    from information_schema.tables
    where table_schema in ('raw','landing','staging','intermediate','marts','features')
    group by 1 order by 1
""")

In [ ]:
# customers by state and access technology (raw source data)
db.query_df("select * from raw.customers limit 5")

If `marts` is empty, build the dbt project once: open **dbt Studio** (`/dbt/`) and press *Build*, or run `cd ~/work/dbt_project && dbt build` in a terminal.

In [ ]:
try:
    display(db.query_df("select * from marts.fct_revenue_monthly order by 1 desc limit 12"))
except Exception as exc:
    print('marts not built yet:', exc)

## 2. The shared lake

`~/work/lake` is the shared folder `/data/lake`; the seed writes Parquet copies of the raw tables there. DuckDB queries files directly.

In [ ]:
import os, duckdb
lake = os.environ.get('LAKE_DIR', '/data/lake')
for root, _, files in os.walk(lake):
    for f in files[:5]:
        print(os.path.join(root, f))

## 3. BigQuery (enterprise EDW)

With an identity assigned in **Studio → Cloud identities** (BigQuery · read), `bsp_common.gcp` builds credentials in memory — pick the identity and project below. Without one, this cell says so and the local warehouse stands in.

In [ ]:
import os
from bsp_common import gcp, pick

sel = pick.gcp_picker(identity=gcp.assigned_name('bigquery_read') or None)
sel.display()
creds = sel.credentials()
if creds:
    from google.cloud import bigquery
    client = bigquery.Client(project=sel.project, credentials=creds)
    print(sel.identity, '·', sel.project, '·', [d.dataset_id for d in client.list_datasets(max_results=20)])
else:
    print('BigQuery not connected — assign a BigQuery identity in Studio → Cloud identities')

## 4. Other services

Each answers only when its capability is running (Studio → Compose).

In [ ]:
import httpx, os
checks = {
    'Trino': 'http://trino:8080/v1/info',
    'Connectors': os.environ.get('CONNECTORS_URL', 'http://connectors:8092/connectors') + '/healthz',
    'LLM gateway': os.environ.get('LLM_GATEWAY_URL', 'http://harness:8096/ai/v1') + '/models',
    'MLflow': os.environ.get('MLFLOW_TRACKING_URI', 'http://mlflow:5000/mlflow') + '/health',
}
for name, url in checks.items():
    try:
        r = httpx.get(url, timeout=3, trust_env=False)
        print(f'{name:12} {r.status_code}')
    except Exception as exc:
        print(f'{name:12} not running ({type(exc).__name__})')

## Next

* Persona notebooks sit next to this one in `notebooks/`.
* **Import & Export** (`/ingress/`) loads files, BigQuery tables or Trino queries into `landing`.
* **Docs** on the Studio launchpad has a guide per appliance.